# 003-pose-search: diagnostics

# Imports

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from uavloc.data.drone_frames import load_frames
from uavloc.geo import LatLon, east_north_m
from uavloc.matching.benchmark import paired_frame_bootstrap, per_frame_error
from uavloc.track import REPO_ROOT, experiment_dir, load_runs

# Params

In [ ]:
EXP = "003-pose-search"
RUNS = ("003.1", "003.2", "003.3", "003.4")
BASELINES = {"001.1": "001-seg-map-correction", "002.2": "002-perfect-satellite-map"}
PAIRS = (("003.2", "003.4"), ("003.1", "002.2"), ("003.2", "002.2"), ("003.3", "001.1"), ("003.4", "001.1"))
POSE_COLUMNS = ("heading_offset_deg", "scale")
POC_RESULTS = "pocs/pose-reregistration/results.csv"
POC_MEDIAN_SCALE = 0.6
PRIMARY_NOISE_M = 15.0
N_BOOTSTRAP = 10000
CI_PCT = 95
SEED = 42
FRAMES_DATASET = "drone-2025-easy"

# Data

In [ ]:
artifacts = experiment_dir(EXP, create=False) / "artifacts"
records = {run_id: pd.read_csv(artifacts / f"records-{run_id}.csv") for run_id in RUNS}
records |= {run_id: pd.read_csv(experiment_dir(exp, create=False) / "artifacts" / f"records-{run_id}.csv")
            for run_id, exp in BASELINES.items()}
{run_id: len(r) for run_id, r in records.items()}

In [ ]:
poc = pd.read_csv(REPO_ROOT / POC_RESULTS).set_index("frame")
truth = {f.name: f.position for f in load_frames(FRAMES_DATASET)}
poc[["block", "yaw_offset_deg", "scale"]].groupby("block").median()

In [ ]:
labels = {r["run"]: f'{r["run"]} (registered {r["config"].get("registered_run", r["run"])})' for r in load_runs(EXP)}
labels

In [ ]:
pose_columns_present = {run_id: all(c in records[run_id].columns for c in POSE_COLUMNS) for run_id in RUNS}
pose_columns_present

# Run

In [ ]:
def primary(run_id: str) -> pd.DataFrame:
    level = records[run_id]
    return level[level["noise_m"] == PRIMARY_NOISE_M].assign(block=lambda d: d["frame"].map(poc["block"]))

In [ ]:
def bias_m(rows: pd.DataFrame) -> tuple[float, float]:
    """Mean (east, north) of corrected minus true position."""
    shifts = [east_north_m(truth[f], LatLon(lat, lon)) for f, lat, lon in zip(rows["frame"], rows["corrected_lat"], rows["corrected_lon"])]
    return tuple(float(v) for v in np.mean(shifts, axis=0))

In [ ]:
summary_rows = []
for run_id in tuple(RUNS) + tuple(BASELINES):
    rows = primary(run_id)
    east, north = bias_m(rows)
    summary_rows.append({"run": run_id, "mean_error_m": rows["error_m"].mean(), "improved_share": (rows["error_m"] < rows["prior_error_m"]).mean(),
                         "mean_peak_z": rows["peak_z"].mean(), "bias_east_m": east, "bias_north_m": north,
                         **{f"mean_error_{n:g}m": records[run_id].loc[records[run_id]["noise_m"] == n, "error_m"].mean()
                            for n in sorted(records[run_id]["noise_m"].unique())}})
summary = pd.DataFrame(summary_rows).set_index("run")
summary.round(3)

In [ ]:
blocks = pd.DataFrame({run_id: per_frame_error(records[run_id], PRIMARY_NOISE_M).groupby(poc["block"]).mean()
                       for run_id in tuple(RUNS) + tuple(BASELINES)})
blocks.insert(0, "frames", poc.groupby("block").size())
blocks.round(2)

# Heading found vs the PoC

In [ ]:
heading_rows = []
for run_id in RUNS:
    rows = primary(run_id).assign(poc_offset_deg=lambda d: d["frame"].map(poc["yaw_offset_deg"]))
    rows["abs_diff_deg"] = (rows["heading_offset_deg"] - rows["poc_offset_deg"]).abs()
    for block, group in rows.groupby("block"):
        heading_rows.append({"run": run_id, "block": block, "draws": len(group),
                             "median_chosen_offset_deg": group["heading_offset_deg"].median(),
                             "median_poc_offset_deg": group["poc_offset_deg"].median(),
                             "median_abs_diff_deg": group["abs_diff_deg"].median()})
    heading_rows.append({"run": run_id, "block": "all", "draws": len(rows),
                         "median_chosen_offset_deg": rows["heading_offset_deg"].median(),
                         "median_poc_offset_deg": rows["poc_offset_deg"].median(),
                         "median_abs_diff_deg": rows["abs_diff_deg"].median()})
heading_vs_poc = pd.DataFrame(heading_rows)
heading_vs_poc.pivot(index="block", columns="run", values="median_abs_diff_deg").round(1)

In [ ]:
heading_vs_poc.pivot(index="block", columns="run", values="median_chosen_offset_deg").assign(
    poc=heading_vs_poc.groupby("block")["median_poc_offset_deg"].first()).round(1)

# Head-to-head

In [ ]:
head_to_head = pd.DataFrame([{"pair": f"{a} - {b}", **paired_frame_bootstrap(records[a], records[b], PRIMARY_NOISE_M,
                                                                              N_BOOTSTRAP, CI_PCT, SEED)}
                             for a, b in PAIRS])
head_to_head.round(3)

# Scale distribution

In [ ]:
scale_shares = pd.DataFrame({run_id: primary(run_id)["scale"].value_counts(normalize=True).sort_index() for run_id in RUNS}).fillna(0)
scale_medians = {run_id: float(primary(run_id)["scale"].median()) for run_id in RUNS}
scale_shares.round(3), scale_medians, POC_MEDIAN_SCALE

# Plots

In [ ]:
fig_heading, axes = plt.subplots(1, len(RUNS), figsize=(4 * len(RUNS), 3.8), sharey=True, squeeze=False)
axes = axes[0]
for ax, run_id in zip(axes, RUNS):
    chosen = primary(run_id).groupby("frame")["heading_offset_deg"].median()
    ax.scatter(poc.loc[chosen.index, "yaw_offset_deg"], chosen, c=poc.loc[chosen.index, "block"], cmap="tab10", vmin=0, vmax=9, s=18)
    ax.plot([-40, 40], [-40, 40], "k:", lw=1)
    ax.set_xlabel("PoC correction, deg"); ax.set_title(labels.get(run_id, run_id), fontsize=9)
axes[0].set_ylabel(f"median chosen offset per frame @ {PRIMARY_NOISE_M:g} m, deg")
fig_heading.suptitle("heading found by the search vs the PoC's per-frame correction (colour: block)", fontsize=9)
fig_heading.tight_layout()

In [ ]:
fig_scale, ax = plt.subplots(figsize=(6, 3.5))
scale_shares.plot.bar(ax=ax)
ax.axvline(list(scale_shares.index).index(POC_MEDIAN_SCALE) if POC_MEDIAN_SCALE in scale_shares.index else 0, color="k", ls=":", lw=1)
ax.set_xlabel("chosen footprint scale"); ax.set_ylabel(f"share of draws @ {PRIMARY_NOISE_M:g} m")
ax.set_title(f"chosen scales (dotted: PoC median {POC_MEDIAN_SCALE:g})", fontsize=9)
fig_scale.tight_layout()

In [ ]:
fig_blocks, ax = plt.subplots(figsize=(7, 3.5))
blocks.drop(columns="frames").plot.bar(ax=ax)
ax.axhline(PRIMARY_NOISE_M, color="k", ls=":", lw=1)
ax.set_xlabel("time block"); ax.set_ylabel(f"mean error @ {PRIMARY_NOISE_M:g} m, m"); ax.legend(fontsize=7)
fig_blocks.tight_layout()

In [ ]:
figures = experiment_dir(EXP) / "figures"
figures.mkdir(exist_ok=True)
fig_heading.savefig(figures / "diagnostics-heading-vs-poc.png", dpi=100)
fig_scale.savefig(figures / "diagnostics-scales.png", dpi=100)
fig_blocks.savefig(figures / "diagnostics-blocks.png", dpi=100)

In [ ]:
diagnostics = {
    "run_labels": labels, "pose_columns_present": pose_columns_present,
    "summary_at_primary_noise": summary.reset_index().to_dict(orient="records"),
    "per_block_mean_error": blocks.reset_index().to_dict(orient="records"),
    "heading_vs_poc": heading_vs_poc.to_dict(orient="records"),
    "head_to_head": head_to_head.to_dict(orient="records"),
    "scale_shares": scale_shares.reset_index().to_dict(orient="records"), "scale_medians": scale_medians,
}
(artifacts / "diagnostics.json").write_text(json.dumps(diagnostics, indent=1, default=float))